# Import Librares

In [1]:
import sqlite3
import csv
import pandas as pd

# Define Data Path

In [2]:
data_path = '../MIMIC_Data/physionet.org/files/mimiciii/1.4/1'

# Create a SQL db given MIMIC csv files

In [3]:
conn = sqlite3.connect('mimic_database.db')
file_names = [
    "ADMISSIONS",
    "CALLOUT",
    "CAREGIVERS",
    "CHARTEVENTS",
    "CPTEVENTS",
    "D_CPT",
    "D_ICD_DIAGNOSES",
    "D_ICD_PROCEDURES",
    "D_ITEMS",
    "D_LABITEMS",
    "DATETIMEEVENTS",
    "DIAGNOSES_ICD",
    "DRGCODES",
    "ICUSTAYS",
    "INPUTEVENTS_CV",
    "INPUTEVENTS_MV",
    "LABEVENTS",
    "MICROBIOLOGYEVENTS",
    "NOTEEVENTS",
    "OUTPUTEVENTS",
    "PATIENTS",
    "PRESCRIPTIONS",
    "PROCEDUREEVENTS_MV",
    "PROCEDURES_ICD",
    "SERVICES",
    "TRANSFERS"
]
for file in file_names:
    for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
        chunk.to_sql(file, conn, index=False, if_exists='append')

conn.close()

C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp\ipykernel_26028\2926986507.py:31: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f"{data_path}/{file}.csv", chunksize=100000):
C:\Users\patel\AppData\Local\Temp

# Create DB Connection

In [2]:
conn = sqlite3.connect('mimic_database.db')
cur = conn.cursor()

# Create SQL Queries

# Demographic Analysis

## Query 1

Patient analysis. Determine the number of male and female patients.

In [5]:
query = '''
            SELECT GENDER, COUNT(*)
            FROM PATIENTS
            GROUP BY GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 20399), ('M', 26121)]


## Query 2

Patient analysis. Determine the age distribution of the patients.

In [6]:
query = '''
            SELECT 
                CASE 
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) < 18 THEN '0-17'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 18 AND 34 THEN '18-34'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 35 AND 49 THEN '35-49'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 50 AND 64 THEN '50-64'
                    WHEN strftime('%Y', DOD) - strftime('%Y', DOB) BETWEEN 65 AND 79 THEN '65-79'
                    WHEN DOD IS NOT NULL THEN '80+'
                    ELSE 'Alive'
                END AS age_group,
                COUNT(*) AS patient_count
            FROM PATIENTS
            GROUP BY age_group
            ORDER BY age_group;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('0-17', 74), ('18-34', 278), ('35-49', 1010), ('50-64', 3023), ('65-79', 5146), ('80+', 6228), ('Alive', 30761)]


## Query 3

Determine the different types of Religions patient practice, and their counts.

In [4]:
query = '''
    SELECT 
        a.RELIGION, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
        FROM ADMISSIONS 
        GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    GROUP BY a.RELIGION
    ORDER BY patient_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('CATHOLIC', 15659), ('NOT SPECIFIED', 9550), ('UNOBTAINABLE', 7711), ('PROTESTANT QUAKER', 5117), ('JEWISH', 3833), ('OTHER', 2104), ('EPISCOPALIAN', 589), (None, 443), ('CHRISTIAN SCIENTIST', 360), ('GREEK ORTHODOX', 323), ('BUDDHIST', 195), ('MUSLIM', 157), ('UNITARIAN-UNIVERSALIST', 104), ("JEHOVAH'S WITNESS", 104), ('HINDU', 101), ('ROMANIAN EAST. ORTH', 66), ('7TH DAY ADVENTIST', 57), ('BAPTIST', 25), ('HEBREW', 15), ('METHODIST', 6), ('LUTHERAN', 1)]


# Query 4

Determine the count of patient's ethnicities

In [13]:
query = '''
    SELECT 
        a.ETHNICITY, 
        COUNT(*) AS patient_count
    FROM ADMISSIONS a
    JOIN (
        SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
        FROM ADMISSIONS 
        GROUP BY SUBJECT_ID
    ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
    GROUP BY a.ETHNICITY
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('WHITE', 32074), ('UNKNOWN/NOT SPECIFIED', 4236), ('BLACK/AFRICAN AMERICAN', 3585), ('HISPANIC OR LATINO', 1350), ('ASIAN', 1304), ('OTHER', 1256), ('UNABLE TO OBTAIN', 792), ('PATIENT DECLINED TO ANSWER', 498), ('ASIAN - CHINESE', 223), ('BLACK/CAPE VERDEAN', 159), ('HISPANIC/LATINO - PUERTO RICAN', 146), ('MULTI RACE ETHNICITY', 111), ('WHITE - RUSSIAN', 105), ('BLACK/HAITIAN', 71), ('WHITE - OTHER EUROPEAN', 69), ('HISPANIC/LATINO - DOMINICAN', 60), ('ASIAN - ASIAN INDIAN', 57), ('AMERICAN INDIAN/ALASKA NATIVE', 45), ('WHITE - BRAZILIAN', 42), ('ASIAN - VIETNAMESE', 41), ('PORTUGUESE', 36), ('BLACK/AFRICAN', 32), ('MIDDLE EASTERN', 28), ('HISPANIC/LATINO - GUATEMALAN', 25), ('WHITE - EASTERN EUROPEAN', 22), ('NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 15), ('HISPANIC/LATINO - CUBAN', 15), ('ASIAN - OTHER', 15), ('ASIAN - FILIPINO', 15), ('HISPANIC/LATINO - SALVADORAN', 14), ('HISPANIC/LATINO - MEXICAN', 11), ('ASIAN - KOREAN', 11), ('ASIAN - CAMBODIAN', 10), ('HISPANIC/LATINO - C

## Query 5

Lab events analysis. Get the lab event's "ITEMID" sorted by the number of times the "FLAG" was "abnormal"

In [6]:
query = '''
            SELECT 
                ITEMID, 
                COUNT(*) AS abnormal_count
            FROM LABEVENTS
            WHERE FLAG = 'abnormal'
            GROUP BY ITEMID
            ORDER BY abnormal_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[(51221, 783689), (51279, 673592), (51222, 667697), (50931, 508914), (51006, 442789), (51277, 346186), (51274, 337209), (50821, 325988), (50912, 321092), (51301, 320247), (51265, 287204), (50893, 268662), (51275, 235026), (51237, 222487), (50970, 219002), (50902, 217537), (50882, 214703), (51248, 208490), (50818, 200752), (50820, 197677), (51249, 157729), (50809, 154439), (50983, 130416), (50804, 129656), (51250, 129401), (51256, 124801), (51244, 115710), (50808, 111158), (50878, 99614), (50971, 95976), (50863, 94076), (50862, 91552), (50861, 89439), (50811, 80401), (50885, 78440), (50813, 77982), (50960, 67117), (50910, 62225), (51003, 59242), (50954, 58100), (50868, 48993), (51493, 38514), (51009, 38333), (50822, 35541), (51516, 23470), (51254, 22654), (50956, 21982), (51200, 21944), (50911, 21896), (51214, 20262), (50883, 19442), (50824, 18770), (51251, 17279), (50867, 17271), (51143, 15623), (51218, 15344), (51257, 14006), (50967, 12688), (50806, 12627), (51144, 12051), (51255, 115

## Query 6

From Query 5, we see that the top lab event that has the highest number of abnormal tests has an ITEMID = 51221

Using the D_LABITEMS table, determine the LABEL associated with ITEMID = 51221

In [7]:
query = '''
            SELECT 
                LABEL
            FROM D_LABITEMS
            WHERE ITEMID = 51221;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Hematocrit',)]


## Query 7

Get the SUBJECT ID of the patients who tested abnormal for the lab test with ITEMID = 51221

In [14]:
query = '''
    SELECT DISTINCT SUBJECT_ID
    FROM LABEVENTS
    WHERE ITEMID = 51221
    AND FLAG = 'abnormal';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[(3,), (2,), (4,), (6,), (7,), (8,), (11,), (9,), (12,), (13,), (17,), (16,), (19,), (20,), (21,), (18,), (22,), (23,), (36,), (26,), (35,), (32,), (27,), (28,), (30,), (33,), (31,), (34,), (25,), (38,), (41,), (42,), (44,), (43,), (64,), (45,), (61,), (52,), (55,), (46,), (59,), (62,), (49,), (54,), (67,), (68,), (37,), (56,), (57,), (80,), (77,), (84,), (93,), (94,), (85,), (69,), (91,), (72,), (81,), (73,), (78,), (71,), (86,), (83,), (74,), (79,), (75,), (107,), (88,), (97,), (100,), (105,), (96,), (92,), (99,), (109,), (101,), (98,), (103,), (108,), (95,), (87,), (115,), (114,), (106,), (113,), (129,), (130,), (112,), (117,), (124,), (125,), (110,), (118,), (119,), (123,), (111,), (133,), (137,), (138,), (141,), (135,), (126,), (132,), (134,), (143,), (145,), (127,), (144,), (131,), (136,), (139,), (140,), (142,), (146,), (148,), (149,), (152,), (156,), (147,), (157,), (155,), (151,), (153,), (154,), (161,), (165,), (150,), (171,), (159,), (158,), (160,), (169,), (174,), (175,), (

## Query 8

Get the gender distribution for the patients who tested abnormal for lab test with ITEMID = 51221

In [13]:
query = '''
            SELECT 
                p.GENDER, 
                COUNT(DISTINCT p.SUBJECT_ID) AS patient_count
            FROM PATIENTS p
            JOIN LABEVENTS l ON p.SUBJECT_ID = l.SUBJECT_ID
            WHERE l.ITEMID = 51221
            AND l.FLAG = 'abnormal'
            GROUP BY p.GENDER;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('F', 17713), ('M', 23086)]


# Query 9

For each drug, get the count of the patients that were prescribed that drug, only including the patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [4]:
query = '''
            SELECT 
                p.DRUG, 
                COUNT(DISTINCT p.SUBJECT_ID) AS prescription_count
            FROM PRESCRIPTIONS p
            JOIN (
                SELECT DISTINCT SUBJECT_ID 
                FROM LABEVENTS 
                WHERE ITEMID = 51221 
                AND FLAG = 'abnormal'
            ) l ON p.SUBJECT_ID = l.SUBJECT_ID
            GROUP BY p.DRUG
            ORDER BY prescription_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Potassium Chloride', 28462), ('Sodium Chloride 0.9%  Flush', 28192), ('Acetaminophen', 27884), ('Magnesium Sulfate', 25635), ('Insulin', 24543), ('Heparin', 23512), ('Docusate Sodium', 22945), ('Iso-Osmotic Dextrose', 20784), ('D5W', 20784), ('Morphine Sulfate', 20315), ('NS', 19950), ('SW', 19634), ('Furosemide', 19475), ('Calcium Gluconate', 19369), ('Bisacodyl', 18212), ('Senna', 16933), ('Pantoprazole', 16748), ('Aspirin', 15502), ('Lorazepam', 14961), ('0.9% Sodium Chloride', 14465), ('Propofol', 13597), ('Vial', 12831), ('Dextrose 50%', 12573), ('Docusate Sodium (Liquid)', 11944), ('Fentanyl Citrate', 11839), ('LR', 11796), ('Oxycodone-Acetaminophen', 11747), ('5% Dextrose', 11656), ('Metoprolol', 11504), ('Ondansetron', 11494), ('Vancomycin', 10737), ('Pantoprazole Sodium', 10352), ('Metoprolol Tartrate', 10118), ('Famotidine', 9956), ('Albuterol 0.083% Neb Soln', 9913), ('Metoclopramide', 9615), ('Atorvastatin', 9529), ('Aspirin EC', 9392), ('Levofloxacin', 9288), ('Ipratrop

## Query 10

Look at the religion distrbution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [ ]:
query = '''
            SELECT 
                a.RELIGION, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                -- Find the first admission per patient
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
                FROM ADMISSIONS 
                GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            WHERE a.SUBJECT_ID IN (
                -- Get unique patients who had an abnormal test for ITEMID 51221
                SELECT DISTINCT SUBJECT_ID 
                FROM LABEVENTS 
                WHERE ITEMID = 51221 
                AND FLAG = 'abnormal'
            )
            GROUP BY a.RELIGION
            ORDER BY patient_count DESC;
        '''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('CATHOLIC', 14143), ('NOT SPECIFIED', 8378), ('UNOBTAINABLE', 6007), ('PROTESTANT QUAKER', 4663), ('JEWISH', 3510), ('OTHER', 1868), ('EPISCOPALIAN', 525), (None, 411), ('CHRISTIAN SCIENTIST', 296), ('GREEK ORTHODOX', 295), ('BUDDHIST', 156), ('MUSLIM', 131), ("JEHOVAH'S WITNESS", 95), ('UNITARIAN-UNIVERSALIST', 89), ('HINDU', 79), ('ROMANIAN EAST. ORTH', 60), ('7TH DAY ADVENTIST', 49), ('BAPTIST', 22), ('HEBREW', 15), ('METHODIST', 6), ('LUTHERAN', 1)]


## Query 11

Look at the ethnicity distrbution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [4]:
query = '''
            SELECT 
                a.ETHNICITY, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                -- Find the first admission per patient
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
                FROM ADMISSIONS 
                GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            WHERE a.SUBJECT_ID IN (
                -- Get unique patients who had an abnormal test for ITEMID 51221
                SELECT DISTINCT SUBJECT_ID 
                FROM LABEVENTS 
                WHERE ITEMID = 51221 
                AND FLAG = 'abnormal'
            )
            GROUP BY a.ETHNICITY
            ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('WHITE', 28422), ('UNKNOWN/NOT SPECIFIED', 3834), ('BLACK/AFRICAN AMERICAN', 3031), ('HISPANIC OR LATINO', 1137), ('OTHER', 1023), ('ASIAN', 918), ('UNABLE TO OBTAIN', 728), ('PATIENT DECLINED TO ANSWER', 424), ('ASIAN - CHINESE', 195), ('BLACK/CAPE VERDEAN', 143), ('HISPANIC/LATINO - PUERTO RICAN', 134), ('WHITE - RUSSIAN', 102), ('MULTI RACE ETHNICITY', 98), ('WHITE - OTHER EUROPEAN', 64), ('BLACK/HAITIAN', 64), ('HISPANIC/LATINO - DOMINICAN', 58), ('ASIAN - ASIAN INDIAN', 54), ('WHITE - BRAZILIAN', 40), ('ASIAN - VIETNAMESE', 35), ('PORTUGUESE', 32), ('BLACK/AFRICAN', 29), ('AMERICAN INDIAN/ALASKA NATIVE', 29), ('MIDDLE EASTERN', 26), ('WHITE - EASTERN EUROPEAN', 20), ('HISPANIC/LATINO - GUATEMALAN', 19), ('HISPANIC/LATINO - CUBAN', 14), ('ASIAN - OTHER', 14), ('NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 13), ('ASIAN - FILIPINO', 13), ('HISPANIC/LATINO - MEXICAN', 11), ('HISPANIC/LATINO - SALVADORAN', 10), ('ASIAN - KOREAN', 10), ('ASIAN - CAMBODIAN', 10), ('HISPANIC/LATINO - CEN

## Query 12

Look at the insurance distribution for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [7]:
query = '''
            SELECT 
                a.INSURANCE, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                -- Find the first admission per patient
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
                FROM ADMISSIONS 
                GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            WHERE a.SUBJECT_ID IN (
                -- Get unique patients who had an abnormal test for ITEMID 51221
                SELECT DISTINCT SUBJECT_ID 
                FROM LABEVENTS 
                WHERE ITEMID = 51221 
                AND FLAG = 'abnormal'
            )
            GROUP BY a.INSURANCE
            ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('Medicare', 19746), ('Private', 15642), ('Medicaid', 3643), ('Government', 1277), ('Self Pay', 491)]


## Query 13

Look at the admission type for patients who were abnormal for the lab test "Hematocrit" with ITEMID = 51221


In [9]:
query = '''
            SELECT 
                a.ADMISSION_TYPE, 
                COUNT(*) AS patient_count
            FROM ADMISSIONS a
            JOIN (
                -- Find the first admission per patient
                SELECT SUBJECT_ID, MIN(ADMITTIME) AS first_admit 
                FROM ADMISSIONS 
                GROUP BY SUBJECT_ID
            ) fa ON a.SUBJECT_ID = fa.SUBJECT_ID AND a.ADMITTIME = fa.first_admit
            WHERE a.SUBJECT_ID IN (
                -- Get unique patients who had an abnormal test for ITEMID 51221
                SELECT DISTINCT SUBJECT_ID 
                FROM LABEVENTS 
                WHERE ITEMID = 51221 
                AND FLAG = 'abnormal'
            )
            GROUP BY a.ADMISSION_TYPE
            ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('EMERGENCY', 29656), ('ELECTIVE', 6209), ('NEWBORN', 3821), ('URGENT', 1113)]


## Query 14

Determine the average length of stay for the admissions where the patient tested abnormal for the lab test "Hematocrit" with ITEMID = 51221 (can split by admission type (EMERGENCY, URGENT, ELECTIVE, etc.))

In [17]:
query = '''
    SELECT 
        AVG(julianday(a.DISCHTIME) - julianday(a.ADMITTIME)) AS overall_avg_length_of_stay
    FROM ADMISSIONS a
    JOIN LABEVENTS l ON a.SUBJECT_ID = l.SUBJECT_ID AND a.HADM_ID = l.HADM_ID
    WHERE l.ITEMID = 51221
    AND l.FLAG = 'abnormal';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[(21.155649549779117,)]


The unit of the average length of stay is days.

## Query 15

Determine the top 10 diagnosis for the admissions where the patient tested abnormal for the lab test "Hematocrit" with ITEMID = 51221

In [11]:
query = '''
    SELECT 
        d.ICD9_CODE, 
        COUNT(DISTINCT d.SUBJECT_ID || '-' || d.HADM_ID) AS diagnosis_count
    FROM DIAGNOSES_ICD d
    JOIN (
        -- Get distinct patient-admission pairs with an abnormal test
        SELECT DISTINCT SUBJECT_ID, HADM_ID 
        FROM LABEVENTS 
        WHERE ITEMID = 51221 
        AND FLAG = 'abnormal'
    ) l ON d.SUBJECT_ID = l.SUBJECT_ID AND d.HADM_ID = l.HADM_ID
    GROUP BY d.ICD9_CODE
    ORDER BY diagnosis_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)


[('4019', 19509), ('4280', 12529), ('42731', 12408), ('41401', 11940), ('5849', 8889), ('25000', 8639), ('2724', 8252), ('51881', 7212), ('5990', 6366), ('53081', 6016), ('2720', 5670), ('2859', 5349), ('2449', 4674), ('486', 4640), ('2851', 4532), ('2762', 4361), ('496', 4215), ('99592', 3846), ('V5861', 3636), ('0389', 3636), ('5070', 3581), ('5859', 3352), ('40390', 3330), ('311', 3225), ('412', 3150), ('3051', 3053), ('2875', 3012), ('41071', 2983), ('2761', 2975), ('V290', 2935), ('V4581', 2923), ('4240', 2810), ('V053', 2713), ('5119', 2682), ('V1582', 2660), ('V4582', 2625), ('40391', 2546), ('78552', 2542), ('4241', 2493), ('V5867', 2441), ('42789', 2339), ('9971', 2279), ('5845', 2265), ('2760', 2213), ('32723', 2201), ('5180', 2113), ('2767', 2093), ('45829', 2080), ('4168', 2050), ('49390', 2021), ('2749', 2015), ('4589', 1959), ('5856', 1867), ('73300', 1837), ('78039', 1819), ('5185', 1772), ('V3000', 1766), ('4271', 1712), ('4111', 1624), ('4254', 1599), ('V1251', 1539), 

## Query 16

Get the short title and the long title of the diagnosis with the highest count for patients who tested abnormal for lab test with ITEMID = 51221

In [19]:
query = '''
    SELECT SHORT_TITLE, LONG_TITLE
    FROM D_ICD_DIAGNOSES
    WHERE ICD9_CODE = '4019';
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('Hypertension NOS', 'Unspecified essential hypertension')]


## Query 17

Determine the types of ICUs

In [7]:
query = '''
    SELECT FIRST_CAREUNIT AS ICU_TYPE FROM ICUSTAYS
    UNION
    SELECT LAST_CAREUNIT FROM ICUSTAYS;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('CCU',), ('CSRU',), ('MICU',), ('NICU',), ('SICU',), ('TSICU',)]


## Query 18

Determine the counts of "FIRST_CAREUNIT" for ICU stays of patients

In [8]:
query = '''
    SELECT FIRST_CAREUNIT, COUNT(*) AS unit_count
    FROM ICUSTAYS
    GROUP BY FIRST_CAREUNIT
    ORDER BY unit_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('MICU', 21088), ('CSRU', 9312), ('SICU', 8891), ('NICU', 8100), ('CCU', 7726), ('TSICU', 6415)]


## Query 19

From Query 18, for the "FIRST_CAREUNIT" with the largest count, determine the length-of-stay distribution.

In [12]:
query = '''
    WITH Most_Common_Unit AS (
        -- Get the most frequent FIRST_CAREUNIT
        SELECT FIRST_CAREUNIT
        FROM ICUSTAYS
        GROUP BY FIRST_CAREUNIT
        ORDER BY COUNT(*) DESC
        LIMIT 1
    )
    SELECT 
        CASE 
            WHEN LOS < 1 THEN '0-1 days'
            WHEN LOS BETWEEN 1 AND 3 THEN '1-3 days'
            WHEN LOS BETWEEN 4 AND 7 THEN '4-7 days'
            WHEN LOS BETWEEN 8 AND 14 THEN '8-14 days'
            ELSE '15+ days'
        END AS LOS_Category,
        COUNT(*) AS patient_count
    FROM ICUSTAYS
    WHERE FIRST_CAREUNIT = (SELECT FIRST_CAREUNIT FROM Most_Common_Unit)
    GROUP BY LOS_Category
    ORDER BY patient_count DESC;
'''
cur.execute(query)
rows = cur.fetchall()
print(rows)

[('1-3 days', 10119), ('0-1 days', 3539), ('15+ days', 3460), ('4-7 days', 2615), ('8-14 days', 1355)]


# Close DB Connection

In [ ]:
conn.close()